# 3D Scan Pipeline (Modular Version)

This notebook runs the 3D scan pipeline step-by-step using modular scripts.
Ensure GPU Accelerator is enabled in Kaggle settings.

In [ ]:
import os

print("⏳ Setting up Environment...")

# 1. Clone Repo
if os.path.exists("3DSCAN"):
    !rm -rf 3DSCAN
!git clone https://github.com/PRIDA-TAKON/3DSCAN.git
os.chdir("3DSCAN")

# 2. Install Dependencies
print("⏳ Installing Dependencies...")
!pip install --upgrade pip
!pip install --upgrade numpy>=2.0 numba scipy pandas scikit-learn opencv-python opencv-python-headless opencv-contrib-python matplotlib pillow plyfile tqdm roma
!pip install taichi

# Install Taichi Splatting from Source
if os.path.exists("taichi-splatting"):
    !rm -rf taichi-splatting
!git clone --depth 1 https://github.com/uc-vision/taichi-splatting.git
!find taichi-splatting -type f \( -name 'pyproject.toml' -o -name 'setup.py' -o -name 'requirements.txt' \) -exec sed -i 's/taichi-nightly/taichi/g' {} +
!pip install ./taichi-splatting

# Install COLMAP & FFmpeg
!apt-get update
!apt-get install -y colmap ffmpeg xvfb

print("✅ Setup Complete.")

In [ ]:
print("=== STEP 1: Extract Frames ===")
# Find video automatically or set manually
import glob
video_path = None
search_paths = ["/kaggle/input", "input"]
for p in search_paths:
    found = glob.glob(f"{p}/**/*.mp4", recursive=True)
    if found:
        video_path = found[0]
        break

if not video_path:
    print("❌ No video found! Please upload a dataset.")
else:
    print(f"🎬 Found video: {video_path}")
    !python scripts/step1_extract_frames.py --input_video "{video_path}" --output_dir "working_data/3d_scan/images"

In [ ]:
print("=== STEP 2: COLMAP SfM ===")
!python scripts/step2_colmap_sfm.py --images_dir "working_data/3d_scan/images" --output_dir "working_data/3d_scan"

In [ ]:
print("=== STEP 3: Train Taichi Splatting ===")
!python scripts/step3_train_splatting.py --project_path "working_data/3d_scan" --output_path "outputs/3d_scan/taichi_splatting"

In [ ]:
print("=== STEP 4: Export ===")
!python scripts/step4_export.py --input_ply "outputs/3d_scan/taichi_splatting/model.ply" --output_splat "outputs/3d_scan/taichi_splatting/model.splat"

In [ ]:
print("=== Compress Output for Download ===")
!zip -r 3d_scan_output.zip outputs/
from IPython.display import FileLink
FileLink(r'3d_scan_output.zip')